# 02. 취약 판정·3축 순위·유형 나누기

**대상**: 실제 매출 기준 취약 후보(이벤트별 판정 → 절반 이상·최소 2번 감소). 이벤트는 **특보가 이어진 기간마다 하나**(나눈 버전)이고, 순위는 **3축 점수만**으로 매깁니다.

**유형 기준 (실행 전에 정함 — 결과를 보고 바꾸지 않음)**

| 기준 | 나누는 법 |
|---|---|
| A. 피해 모양 | 같은 지역·기상 후보 안에서 Impact 점수와 Cumulative 점수(0~1, 클수록 취약)를 각각 0.5로 나눔 → 심각형(둘 다 높음) / 급락형(Impact만) / 지속 침체형(Cumulative만) / 상대적 완만형. Recovery가 3일 이상(“이상” 포함)이면 `회복 지연` 꼬리표 |
| B. 시간대 피해 모양 | 이벤트 기간 시간대별 변화율 $r=\sum(S-B)/\sum B$. 낮 = 06~17시, 저녁 = 18~23시(00~05시는 참고만). 낮이 감소일 때 저녁이 낮 감소폭의 30% 이상 **증가** → 시간 이동형 / 저녁도 낮 감소폭의 50% 이상 **감소** → 전일 감소형(저녁 감소가 낮의 1.5배 이상이면 저녁 집중 감소형) / 그 사이 → 낮 집중 감소형. 낮이 감소가 아니면 저녁 감소형 또는 낮 감소 없음. 낮·저녁 모두 변화가 5% 미만이면 `시간대 차이 작음`. **저녁 칸이 없는 업종**(저녁 매출이 거의 없어 전처리에서 빠짐)은 영업시간 전체가 낮이므로 낮이 5% 이상 감소면 `전일 감소형`, 아니면 `시간대 차이 작음`(비고: 저녁 영업 없음). 반대로 **낮 칸이 없는 업종**(야간 영업)은 저녁 기준으로 같은 규칙 |
| C. 업종 성격 | 업종 이름 규칙표(아래 셀에서 수정 가능). 안 맞는 업종은 `기타`로 두고 목록을 출력 |

**최종 유형 = 시간대 유형(B) × 업종 성격(C)**, 피해 모양(A)은 심각도 꼬리표. 피해 모양은 **후보끼리의 상대 비교**라 '상대적 완만형'도 취약 후보입니다.

**보조 표시 (팀원 검토 반영 — 주 판정은 바꾸지 않음)**
- `경계`: 비율로 판정하는 유형인데 낮 변화가 5% 미만이면 '경계 사례' (저녁이 조금만 움직여도 유형이 바뀔 수 있음)
- `시간대유형_보조5%`: 감소 기준을 −5%로 통일한 판정(낮이 −5%보다 덜 줄면 '낮 감소 없음'/'저녁 감소형' 쪽)
- `저녁증가÷낮감소`, `시간이동_금액기준`: 저녁이 늘어난 업종에서 저녁 증가액(원)이 낮 감소액(원)의 30% 이상인지
- `시간대묶음`: 보고서용 — 낮 집중 / 하루 전체 / 저녁 쪽(저녁 집중·저녁 감소·시간 이동) / 차이 작음

In [ ]:
# 0. 설정
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/big_contest')
try:
    from google.colab import drive; drive.mount('/content/drive')
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
except ImportError:
    pass
import numpy as np, pandas as pd, warnings, matplotlib.pyplot as plt
try:
    import koreanize_matplotlib   # 한글 글꼴
except ImportError:
    print('※ 한글 글꼴 패키지가 없어 그래프 글자가 깨질 수 있습니다')
warnings.filterwarnings('ignore', category=RuntimeWarning); warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

def find(name):
    hits = sorted(ROOT.rglob(name), key=lambda p: p.stat().st_mtime)
    assert hits, f'{name} 을(를) {ROOT} 아래에서 찾지 못했습니다'
    print('읽는 파일:', hits[-1]); return hits[-1]
V4_PATH = find('38_최종baseline_V4.csv')
OUT = ROOT/'results'/'유형나누기'; OUT.mkdir(parents=True, exist_ok=True)

SPECIAL2_DAYS = pd.DatetimeIndex(np.concatenate([pd.date_range(a, b) for a, b in
                 [('2025-08-14', '2025-08-18'), ('2025-10-02', '2025-10-10'), ('2025-12-24', '2025-12-26')]]))
SCOPE_CHECK = ['보험', '통신요금(이동,시내전화)', '결제대행(PG)']
GAP_MERGE = 0                 # 0 = 특보가 이어진 기간마다 하나의 이벤트(나눈 버전)
N_DRAW = 1000                 # 평시 범위 추출 횟수
RANGE_Q, REC_MAX, SEED = (5, 95), 14, 42
REPEAT_MIN, REPEAT_SHARE = 2, 0.5
# 유형 기준
MOVE, ALLDAY, EVE_FOCUS, REC_SLOW, MIN_EFFECT = 0.30, 0.50, 1.5, 3, 0.05
SHIFT_AMT = 0.30            # 보조: 시간 이동을 금액으로 볼 때 저녁 증가액 ÷ 낮 감소액 기준
DAY_SLOTS, EVE_SLOTS = ['06_11', '12_17'], ['18_23']

# 업종 성격 규칙표: 업종 이름에 키워드가 들어 있으면 그 성격 (위에서부터 먼저 맞는 것)
NATURE_RULES = [
    ('야외·이동 의존형', ['화원', '오토바이', '자동차', '스포츠', '레저', '주유', 'LPG', '골프', '고속버스', '철도', '택시', '주차', '캠핑', '자전거']),
    ('예약·행사형', ['예식', '결혼', '호텔', '콘도', '여행', '항공', '장례', '연회', '숙박']),
    ('방문 서비스형', ['세탁', '싸우나', '사우나', '목욕', '노래방', '미용', '피부', '네일', '안마', '헬스', '수영', '당구', 'PC', '오락', '영화', '공연']),
    ('대형 유통형', ['백화점', '쇼핑몰', '기타유통', '할인점', '슈퍼마켓', '양판점', '면세']),
    ('음식·생활 밀착형', ['한식', '일식', '중식', '양식', '요식', '제과', '커피', '편의점', '약국', '주류', '식품', '정육', '농수산', '패스트푸드', '분식']),
    ('구매형 소매', ['의복', '의류', '신발', '시계', '귀금속', '인테리어', '건축자재', '주방', '가구', '문화용품', '수제용품', '서적', '문구', '화장품', '안경', '가전', '컴퓨터', '소프트웨어', '중고', '악기', '음반', '완구']),
    ('교육·의료형', ['학원', '학습지', '유치원', '병원', '의원', '의료', '치과', '한의원', '동물병원']),
    ('업무·기타 서비스', ['연구', '번역', '보험', '통신', '결제대행', '상품권', '복권', '부동산', '법률', '회계']),
]
def nature(ind):
    for name, keys in NATURE_RULES:
        if any(k in ind for k in keys): return name
    return '기타'

# 그래프 색 (검증된 기본 팔레트: 범주 1~4, 감소/증가 = 빨강/파랑, 보조 잉크는 회색)
COL = dict(cat=['#2a78d6', '#eb6834', '#1baf7a', '#eda100'], down='#e34948', up='#2a78d6',
           ink='#0b0b0b', ink2='#52514e', grid='#e4e3df', shade='#ecebe7')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': COL['ink2'],
                     'axes.labelcolor': COL['ink2'], 'xtick.color': COL['ink2'], 'ytick.color': COL['ink2'],
                     'axes.grid': True, 'grid.color': COL['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'axes.unicode_minus': False, 'figure.dpi': 110})

In [ ]:
# 1. 데이터
P = pd.read_csv(V4_PATH, encoding='utf-8-sig', parse_dates=['날짜'])
P['특수일2'] = P['날짜'].isin(SPECIAL2_DAYS); P['특보일'] = (P['폭염등급'] > 0) | (P['호우등급'] > 0)
ALL_DATES = pd.date_range(P['날짜'].min(), P['날짜'].max())
dd = P.drop_duplicates(['대상', '날짜']).set_index(['대상', '날짜']).sort_index()
D = (P[~P['휴무']].groupby(['대상', '업종', 'gkey', '날짜'])
     .agg(S=('S', 'sum'), B=('B', 'sum'), 평시=('평시', 'first'), 특수일2=('특수일2', 'first')).reset_index())
D['y_day'] = (D['S'] - D['B']) / D['B'].replace(0, np.nan); D['요일'] = D['날짜'].dt.dayofweek
print(f'{len(D):,}행, 업종 {D["gkey"].nunique()}개')

In [ ]:
# 2. 이벤트 → 판정 → 반복 → 3축 순위 (재점검2와 같은 규칙, 나눈 버전)
def runs(s):
    out, cur = [], []
    for d, v in s.items():
        if v: cur.append(d)
        elif cur: out.append(cur); cur = []
    if cur: out.append(cur)
    return out
def judge(rate, lo, hi):
    if not np.isfinite([rate, lo, hi]).all(): return '계산 불가'
    return '감소' if rate < lo else ('증가' if rate > hi else '판단 불가')

EP = []
for reg in sorted(P['대상'].unique()):
    w = dd.loc[reg]
    for typ, col in [('폭염', '폭염등급'), ('호우', '호우등급')]:
        s = w[col].reindex(ALL_DATES).fillna(0) > 0; merged = []
        for r in runs(s):
            if merged and (r[0] - merged[-1][-1]).days - 1 <= GAP_MERGE: merged[-1] = list(pd.date_range(merged[-1][0], r[-1]))
            else: merged.append(list(r))
        for i, days in enumerate(merged, 1):
            EP.append(dict(대상=reg, 종류=typ, 에피소드=f'{typ}{i}', 시작=days[0], 종료=days[-1], 날짜=days,
                           분석일=[d for d in days if d not in SPECIAL2_DAYS]))
EP = pd.DataFrame(EP)
print('이벤트 수:', EP.groupby(['대상', '종류']).size().to_dict())

SPAN = {r: set(sum(EP.loc[EP['대상'] == r, '날짜'].tolist(), [])) for r in EP['대상'].unique()}
inspan = np.array([d in SPAN.get(r, ()) for r, d in zip(D['대상'], D['날짜'])])
pool = D[D['평시'].to_numpy() & ~D['특수일2'].to_numpy() & ~inspan & (D['B'] > 0).to_numpy()]
WD = {g: {w_: q for w_, q in z.groupby('요일')} for g, z in pool.groupby('gkey')}
REC_THR = pool.groupby('gkey')['y_day'].quantile(.25).clip(upper=-.10)
ALERT = {r: set(dd.loc[r].index[dd.loc[r]['특보일']]) | SPAN[r] for r in SPAN}
LAST = D.groupby('gkey')['날짜'].max(); Y = D.set_index(['gkey', '날짜'])['y_day']
rng = np.random.default_rng(SEED); rng_i = np.random.default_rng(SEED + 2)

def rng_iid(g, dates):
    ds = np.zeros(N_DRAW); bs = np.zeros(N_DRAW)
    for d in dates:
        q = WD[g].get(d.dayofweek)
        if q is None or q.empty: return np.nan, np.nan
        ix = rng.integers(0, len(q), N_DRAW); ds += (q['S'] - q['B']).to_numpy()[ix]; bs += q['B'].to_numpy()[ix]
    return tuple(np.percentile(ds / bs, RANGE_Q))
def imp_null(g, full, valid, win):
    a = np.full((len(full), N_DRAW), np.nan)
    for j, d in enumerate(full):
        q = WD[g].get(d.dayofweek)
        if d in valid and q is not None and not q.empty: a[j] = q['y_day'].to_numpy()[rng_i.integers(0, len(q), N_DRAW)]
    ws = [np.mean(a[j:j + win], axis=0) for j in range(len(a) - win + 1) if np.isfinite(a[j:j + win]).all()]
    return np.percentile(np.min(ws, axis=0), 5) if ws else np.nan
def recovery(g, reg, end):
    prev = None
    def lower(k):
        lb = (k - 1) - (1 if prev is not None and prev == end + pd.Timedelta(days=k - 1) else 0)
        return float(lb) if lb > 0 else np.nan
    for k in range(1, REC_MAX + 1):
        d = end + pd.Timedelta(days=k)
        if d > LAST[g] or d in ALERT[reg]: return lower(k), '이상'
        v = Y.get((g, d), np.nan)
        if d in SPECIAL2_DAYS or not np.isfinite(v): prev = None; continue
        if v >= REC_THR.get(g, -.10):
            if prev is not None and (d - prev).days == 1: return float(k - 1), '확인'
            prev = d
        else: prev = None
    return lower(REC_MAX + 1), '이상'

rows = []
for _, e in EP.iterrows():
    full = pd.DatetimeIndex(e['날짜']); an = set(e['분석일'])
    sub = D[(D['대상'] == e['대상']) & D['날짜'].isin(an) & (D['B'] > 0) & np.isfinite(D['y_day'])]
    for g, z in sub.groupby('gkey'):
        if g not in WD: continue
        z = z.sort_values('날짜'); win = min(3, len(full)); lo, hi = rng_iid(g, list(z['날짜']))
        delta = (z['S'] - z['B']).to_numpy(); rate = delta.sum() / z['B'].sum()
        yy = pd.Series(delta / z['B'].to_numpy(), index=z['날짜']).reindex(full)
        imp = yy.rolling(win, min_periods=win).mean().min(); il = imp_null(g, full, set(z['날짜']), win)
        rec, st = recovery(g, e['대상'], e['종료'])
        rows.append(dict(대상=e['대상'], 종류=e['종류'], 에피소드=e['에피소드'], gkey=g, 판정=judge(rate, lo, hi), 차이율=rate,
                         평시하한=lo, 평시상한=hi, Impact=imp, Cumulative=delta.sum() / z['B'].mean(), 차액=delta.sum(),
                         회복=0.0 if imp >= il else rec, 이상=(imp < il) and st == '이상' and np.isfinite(rec)))
EPI = pd.DataFrame(rows)

out = []
for (reg, typ, g), z in EPI.groupby(['대상', '종류', 'gkey']):
    ok = z['판정'] != '계산 불가'; dec = z[z['판정'] == '감소']; n, k = int(ok.sum()), len(dec)
    scope = g.split('|')[1] in SCOPE_CHECK; cand = k >= REPEAT_MIN and n > 0 and k / n >= REPEAT_SHARE
    rec = dec['회복'].mean() if dec['회복'].notna().any() else np.nan; lb = bool(dec['이상'].any())
    out.append(dict(대상=reg, 종류=typ, gkey=g, 업종=g.split('|')[1], 반복=f'{k}/{n}', 감소수=k, 이벤트수=n,
                    전체판정='분석범위 보류' if scope and cand else '취약 후보' if cand else '감소 관찰(반복 확인 필요)' if k else '구분 어려움',
                    후보=cand and not scope, Impact=z['Impact'].min(), Cumulative=z['Cumulative'].sum(), 차액=z['차액'].sum(),
                    Recovery=rec, Recovery이상=lb,
                    Recovery표시='정보 없음' if not np.isfinite(rec) else f"{rec:.1f}일" + (' 이상' if lb else '')))
S = pd.DataFrame(out)
for c in ['I점수', 'C점수', 'R점수', '점수', '순위']: S[c] = np.nan
for _, g in S.groupby(['대상', '종류']):
    g = g[g['후보']]
    S.loc[g.index, 'I점수'] = g['Impact'].rank(ascending=False, pct=True)
    S.loc[g.index, 'C점수'] = g['Cumulative'].rank(ascending=False, pct=True)
    S.loc[g.index, 'R점수'] = g['Recovery'].rank(ascending=True, pct=True)
    sc = S.loc[g.index, ['I점수', 'C점수', 'R점수']].mean(axis=1, skipna=True).round(12)
    S.loc[g.index, '점수'] = sc; S.loc[g.index, '순위'] = sc.rank(ascending=False, method='min')
CAND = S[S['후보']].copy()
print('취약 후보 수:', CAND.groupby(['대상', '종류']).size().to_dict())

In [ ]:
# 3. 유형 A — 피해 모양
def shape(r):
    hi_i, hi_c = r['I점수'] > 0.5, r['C점수'] > 0.5
    return '심각형' if hi_i and hi_c else '급락형' if hi_i else '지속 침체형' if hi_c else '상대적 완만형'
CAND['피해모양'] = CAND.apply(shape, axis=1)
CAND['회복지연'] = np.where(CAND['Recovery'] >= REC_SLOW, '회복 지연', '')

# 4. 유형 B — 시간대 피해 모양 (이벤트 분석일 전체, 휴무 칸 제외)
an_days = {(r, t): set(sum(z['분석일'].tolist(), [])) for (r, t), z in EP.groupby(['대상', '종류'])}
slot_rows = []
for _, c in CAND.iterrows():
    z = P[(P['gkey'] == c['gkey']) & P['날짜'].isin(an_days[(c['대상'], c['종류'])]) & ~P['휴무']]
    agg = z.groupby('slot')[['S', 'B']].sum(); rate = (agg['S'] - agg['B']) / agg['B']
    part = lambda sl: (lambda q: (q['S'] - q['B']).sum() / q['B'].sum() if q['B'].sum() > 0 else np.nan)(agg.loc[agg.index.isin(sl)])
    amt = lambda sl: (agg.loc[agg.index.isin(sl), 'S'] - agg.loc[agg.index.isin(sl), 'B']).sum() if agg.index.isin(sl).any() else np.nan
    slot_rows.append(dict(대상=c['대상'], 종류=c['종류'], gkey=c['gkey'], 낮=part(DAY_SLOTS), 저녁=part(EVE_SLOTS),
                          낮차액=amt(DAY_SLOTS), 저녁차액=amt(EVE_SLOTS),
                          **{f'r_{s}': rate.get(s, np.nan) for s in sorted(P['slot'].unique())}))
SL = pd.DataFrame(slot_rows)
def timetype(r):
    d, e = r['낮'], r['저녁']
    if not np.isfinite(d) and not np.isfinite(e): return '계산 불가'
    if not np.isfinite(d):   # 낮 시간대 매출이 거의 없어 빠진 업종(야간 영업) → 저녁 기준
        return '전일 감소형' if e <= -MIN_EFFECT else '시간대 차이 작음'
    if not np.isfinite(e):   # 저녁 시간대 매출이 거의 없어 전처리에서 빠진 업종 → 영업시간 전체가 낮
        return '전일 감소형' if d <= -MIN_EFFECT else '시간대 차이 작음'
    if max(abs(d), abs(e)) < MIN_EFFECT: return '시간대 차이 작음'
    if d >= 0: return '저녁 감소형' if e < 0 else '낮 감소 없음'
    if e >= MOVE * abs(d): return '시간 이동형'
    if e < 0 and abs(e) >= EVE_FOCUS * abs(d): return '저녁 집중 감소형'
    if e <= -ALLDAY * abs(d): return '전일 감소형'
    return '낮 집중 감소형'
SL['시간대유형'] = SL.apply(timetype, axis=1)
SL['비고'] = np.where(SL['저녁'].isna() & SL['낮'].notna(), '저녁 영업 없음(낮 기준 판정)',
             np.where(SL['낮'].isna() & SL['저녁'].notna(), '낮 영업 없음(저녁 기준 판정)', ''))
# 보조 1 — 팀원 제안: 낮이 −5%보다 덜 줄면 '낮 감소 없음' 쪽으로 (감소 기준을 5%로 통일)
def timetype_aux(r):
    d, e = r['낮'], r['저녁']
    if not (np.isfinite(d) and np.isfinite(e)): return timetype(r)
    if max(abs(d), abs(e)) < MIN_EFFECT: return '시간대 차이 작음'
    if d > -MIN_EFFECT: return '저녁 감소형' if e <= -MIN_EFFECT else '낮 감소 없음'
    return timetype(r)
SL['시간대유형_보조5%'] = SL.apply(timetype_aux, axis=1)
# 경계 사례: 비율로 판정하는 유형인데 낮 변화가 5% 미만 → 저녁이 조금만 움직여도 유형이 바뀔 수 있음
ratio_types = ['시간 이동형', '저녁 집중 감소형', '전일 감소형', '낮 집중 감소형', '저녁 감소형', '낮 감소 없음']
SL['경계'] = np.where(SL['시간대유형'].isin(ratio_types) & SL['저녁'].notna() & (SL['낮'].abs() < MIN_EFFECT),
                      '경계 사례(낮 변화 5% 미만)', '')
# 보조 2 — 금액 기준 시간 이동: 저녁 증가액이 낮 감소액을 얼마나 메웠나
SL['저녁증가÷낮감소'] = np.where((SL['낮차액'] < 0) & (SL['저녁차액'] > 0), SL['저녁차액'] / -SL['낮차액'], np.nan)
SL['시간이동_금액기준'] = np.where(SL['저녁증가÷낮감소'] >= SHIFT_AMT, '이동(금액 30% 이상)',
                          np.where(SL['저녁증가÷낮감소'].notna(), '일부만 이동', ''))
# 보고서용 묶음
GROUP = {'낮 집중 감소형': '낮 집중', '전일 감소형': '하루 전체', '저녁 집중 감소형': '저녁 쪽', '저녁 감소형': '저녁 쪽',
         '시간 이동형': '저녁 쪽', '시간대 차이 작음': '차이 작음'}
SL['시간대묶음'] = SL['시간대유형'].map(GROUP).fillna('확인 필요')
CAND = CAND.merge(SL, on=['대상', '종류', 'gkey'], how='left')

# 5. 유형 C — 업종 성격
CAND['업종성격'] = CAND['업종'].map(nature)
etc = sorted(CAND.loc[CAND['업종성격'] == '기타', '업종'].unique())
print('업종 성격이 "기타"로 남은 업종 (필요하면 NATURE_RULES에 추가):', etc if etc else '없음')

CAND['최종유형'] = CAND['업종성격'] + ' × ' + CAND['시간대유형']
print('\n[시간대 유형 분포]'); print(pd.crosstab([CAND['대상'], CAND['종류']], CAND['시간대유형']).to_string())
print('\n[피해 모양 분포]'); print(pd.crosstab([CAND['대상'], CAND['종류']], CAND['피해모양']).to_string())
print('\n[보고서용 시간대 묶음]'); print(pd.crosstab([CAND['대상'], CAND['종류']], CAND['시간대묶음']).to_string())
q = CAND[(CAND['경계'] != '') | (CAND['시간대유형'] != CAND['시간대유형_보조5%']) | (CAND['시간이동_금액기준'] != '')].copy()
for c in ['낮', '저녁']: q[c] = (q[c] * 100).round(1)
q['저녁증가÷낮감소'] = (q['저녁증가÷낮감소'] * 100).round(0)
print('\n[주의 사례 — 경계, 5% 기준 보조 판정이 다른 업종, 저녁이 늘어난 업종의 금액 기준 이동]  (낮·저녁 %, 저녁증가÷낮감소 %)')
print('없음' if q.empty else q[['대상', '종류', '업종', '낮', '저녁', '시간대유형', '시간대유형_보조5%', '경계', '저녁증가÷낮감소', '시간이동_금액기준']].to_string(index=False))
nx = CAND[CAND['시간대유형'] == '계산 불가']
print('\n시간대 유형 계산 불가:', '없음' if nx.empty else nx[['대상', '종류', '업종']].values.tolist())

## 결과 표 — 지역·기상별 취약 후보 (3축 순위순)
- Impact: 이벤트 중 3일 평균 최저 변화율, Cumulative: 누적 감소(평소 하루 매출 며칠치)
- 낮·저녁: 이벤트 기간 그 시간대 매출이 평소보다 몇 % 달라졌나

In [ ]:
# 6. 결과 표
SHOW = ['순위', '업종', '반복', 'Impact', 'Cumulative', 'Recovery표시', '피해모양', '회복지연', '낮', '저녁', '시간대유형', '시간대묶음', '업종성격', '비고', '경계']
AUX = ['시간대유형_보조5%', '저녁증가÷낮감소', '시간이동_금액기준', '낮차액', '저녁차액']
TBL = CAND.sort_values(['대상', '종류', '순위', 'gkey']).copy()
for c in ['Impact', '낮', '저녁']: TBL[c] = (TBL[c] * 100).round(1)
TBL['Cumulative'] = TBL['Cumulative'].round(2)
for (reg, typ), t in TBL.groupby(['대상', '종류']):
    print(f'\n■ {reg} {typ}  (Impact·낮·저녁 %, Cumulative 하루치)'); print(t[SHOW].to_string(index=False))

with pd.ExcelWriter(OUT/'유형나누기_결과.xlsx') as xw:
    TBL[['대상', '종류'] + SHOW + ['최종유형', '차액'] + AUX + [c for c in TBL if c.startswith('r_')]].to_excel(xw, sheet_name='취약후보_유형', index=False)
    pd.crosstab([CAND['대상'], CAND['종류'], CAND['업종성격']], CAND['시간대유형']).reset_index().to_excel(xw, sheet_name='성격x시간대', index=False)
    S.to_excel(xw, sheet_name='전체업종_판정', index=False)
    EPI.to_excel(xw, sheet_name='이벤트별', index=False)
    pd.DataFrame([(n, ', '.join(k)) for n, k in NATURE_RULES], columns=['업종성격', '키워드']).to_excel(xw, sheet_name='업종성격_규칙', index=False)
print('\n저장:', OUT/'유형나누기_결과.xlsx')

## 그래프로 보기
1. **시간대 유형 예시**: 유형마다 대표 업종(그 유형에서 3축 순위가 가장 높은 업종)의 시간대별 변화율
2. **피해 모양 지도**: 후보마다 Impact 점수 × Cumulative 점수, 0.5 선으로 네 칸
3. **피해 모양 예시**: 심각형·급락형·지속 침체형 대표 업종의 하루 변화율 흐름(회색 띠 = 이벤트 기간)
4. **업종 성격 × 시간대 유형**: 어떤 성격의 업종이 어떤 시간대 피해를 보이는지

In [ ]:
# 7. 그래프 1 — 시간대 유형 예시 (폭염·호우 각각)
SLOT_ORDER = sorted(P['slot'].unique(), key=lambda s: int(s.split('_')[0]))
def slot_bars(ax, r, title):
    v = np.array([r.get(f'r_{s}', np.nan) for s in SLOT_ORDER]) * 100
    ax.bar(SLOT_ORDER, np.nan_to_num(v), color=[COL['down'] if x < 0 else COL['up'] for x in v], width=0.6)
    ax.axhline(0, color=COL['ink2'], lw=1)
    for i, x in enumerate(v):
        if np.isfinite(x): ax.text(i, x + (1.5 if x >= 0 else -1.5), f'{x:+.0f}%', ha='center', va='bottom' if x >= 0 else 'top', fontsize=9, color=COL['ink'])
        else: ax.text(i, 0.5, '자료 없음', ha='center', va='bottom', fontsize=8, color=COL['ink2'])
    ax.set_title(title, fontsize=10, color=COL['ink']); ax.set_ylabel('평소 대비 변화율(%)')
    lim = np.nanmax(np.abs(v)) * 1.35 if np.isfinite(v).any() else 10; ax.set_ylim(-lim, lim)

for typ in ['폭염', '호우']:
    g = CAND[CAND['종류'] == typ]
    reps = g.sort_values(['순위', 'gkey']).groupby('시간대유형').head(1)
    reps = reps[reps['시간대유형'] != '계산 불가']
    if reps.empty: continue
    fig, axes = plt.subplots(1, len(reps), figsize=(3.6 * len(reps), 3.4), squeeze=False)
    for ax, (_, r) in zip(axes[0], reps.iterrows()):
        slot_bars(ax, r, f"{r['시간대유형']}\n{r['대상'][:2]} {r['업종']} ({int(r['순위'])}위)")
    fig.suptitle(f'{typ} — 시간대 유형별 대표 업종 (이벤트 기간 시간대별 변화율)', fontsize=11, color=COL['ink'], y=1.04)
    fig.tight_layout(); fig.savefig(OUT/f'그래프1_시간대유형_{typ}.png', bbox_inches='tight'); plt.show()

In [ ]:
# 8. 그래프 2 — 피해 모양 지도
groups = list(CAND.groupby(['대상', '종류']))
fig, axes = plt.subplots(1, len(groups), figsize=(4.2 * len(groups), 4.2), squeeze=False)
for ax, ((reg, typ), g) in zip(axes[0], groups):
    ax.scatter(g['I점수'], g['C점수'], s=46, color=COL['cat'][0], edgecolor='white', linewidth=1.5, zorder=3)
    ax.axvline(0.5, color=COL['ink2'], lw=1, ls='--'); ax.axhline(0.5, color=COL['ink2'], lw=1, ls='--')
    for x, y, t in [(0.98, 0.98, '심각형'), (0.98, 0.02, '급락형'), (0.02, 0.98, '지속 침체형'), (0.02, 0.02, '상대적 완만형')]:
        ax.text(x, y, t, transform=ax.transAxes, ha='right' if x > .5 else 'left', va='top' if y > .5 else 'bottom', fontsize=9,
                color=COL['ink2'], bbox=dict(facecolor='white', edgecolor='none', alpha=.8, pad=1.5), zorder=2)
    for _, r in g.nsmallest(5, '순위').iterrows():
        ax.annotate(r['업종'], (r['I점수'], r['C점수']), xytext=(4, 4), textcoords='offset points', fontsize=8, color=COL['ink'])
    ax.set_xlim(0, 1.12); ax.set_ylim(0, 1.12); ax.set_xlabel('Impact 점수 (깊이, 클수록 취약)'); ax.set_ylabel('Cumulative 점수 (누적, 클수록 취약)')
    ax.set_title(f'{reg[:2]} {typ} (후보 {len(g)}개, 이름은 상위 5)', fontsize=10, color=COL['ink'])
fig.tight_layout(); fig.savefig(OUT/'그래프2_피해모양지도.png', bbox_inches='tight'); plt.show()

In [ ]:
# 9. 그래프 3 — 피해 모양 예시: 하루 변화율 흐름
def timeline(ax, r):
    g, reg, typ = r['gkey'], r['대상'], r['종류']
    eps = EP[(EP['대상'] == reg) & (EP['종류'] == typ)]
    lo, hi = eps['시작'].min() - pd.Timedelta(days=7), eps['종료'].max() + pd.Timedelta(days=14)
    y = D[(D['gkey'] == g) & D['날짜'].between(lo, hi)].set_index('날짜')['y_day'].reindex(pd.date_range(lo, hi)) * 100
    y[y.index.isin(SPECIAL2_DAYS)] = np.nan
    for _, e in eps.iterrows():
        ax.axvspan(e['시작'], e['종료'] + pd.Timedelta(days=1), color=COL['shade'], zorder=0, lw=0)
    ax.plot(y.index, y.values, color=COL['cat'][0], lw=2, zorder=3)
    ax.axhline(0, color=COL['ink2'], lw=1)
    ax.set_ylabel('평소 대비(%)')
    ax.set_title(f"{r['피해모양']} — {reg[:2]} {typ} {r['업종']} ({int(r['순위'])}위, 반복 {r['반복']}, Impact {r['Impact']*100:.0f}%, 누적 {r['Cumulative']:.1f}일치)",
                 fontsize=10, color=COL['ink'], loc='left')

ex = CAND[CAND['종류'] == '폭염'].sort_values(['순위', 'gkey']).groupby('피해모양').head(1)
ex = ex.set_index('피해모양').reindex([k for k in ['심각형', '급락형', '지속 침체형', '상대적 완만형'] if k in set(ex['피해모양'])]).reset_index()
if len(ex):
    fig, axes = plt.subplots(len(ex), 1, figsize=(11, 2.6 * len(ex)), squeeze=False, sharex=True)
    for ax, (_, r) in zip(axes[:, 0], ex.iterrows()): timeline(ax, r)
    axes[0, 0].text(1.0, 1.25, '회색 띠 = 폭염 이벤트 기간, 끊긴 구간 = 특수일(제외)', transform=axes[0, 0].transAxes, ha='right', fontsize=9, color=COL['ink2'])
    fig.tight_layout(); fig.savefig(OUT/'그래프3_피해모양_예시_폭염.png', bbox_inches='tight'); plt.show()

In [ ]:
# 10. 그래프 4 — 업종 성격 × 시간대 유형 (업종 수)
TT_ALL = ['낮 집중 감소형', '시간 이동형', '전일 감소형', '저녁 집중 감소형', '저녁 감소형', '낮 감소 없음', '시간대 차이 작음', '계산 불가']
TT_COL = dict(zip(TT_ALL, COL['cat'] + ['#e87ba4', '#6250d6', '#b5b3ab', '#d9d8d3']))   # 유형마다 색 고정
TT = [t for t in TT_ALL if t in set(CAND['시간대유형'])]
for typ in ['폭염', '호우']:
    ct = pd.crosstab(CAND.loc[CAND['종류'] == typ, '업종성격'], CAND.loc[CAND['종류'] == typ, '시간대유형']).reindex(columns=TT, fill_value=0)
    ct = ct.loc[:, ct.sum() > 0]   # 이 그래프에 없는 유형은 범례에서 뺌
    if ct.empty: continue
    ct = ct.loc[ct.sum(axis=1).sort_values().index]
    colors = [TT_COL[c] for c in ct.columns]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(ct) + 1.8))
    left = np.zeros(len(ct))
    for c, colr in zip(ct.columns, colors):
        ax.barh(ct.index, ct[c], left=left, color=colr, edgecolor='white', linewidth=2, label=c)
        for i, (l, v) in enumerate(zip(left, ct[c])):
            if v > 0: ax.text(l + v / 2, i, str(v), ha='center', va='center', fontsize=9, color=COL['ink'], bbox=dict(facecolor='white', edgecolor='none', alpha=.85, pad=1))
        left += ct[c].to_numpy()
    ax.set_xlabel('업종 수 (두 지역 합)'); ax.xaxis.get_major_locator().set_params(integer=True)
    ax.legend(ncol=min(4, len(ct.columns)), loc='upper left', bbox_to_anchor=(0, -0.18), frameon=False, fontsize=9)
    ax.grid(axis='y', visible=False); ax.set_title(f'{typ} — 업종 성격별 시간대 피해 유형', fontsize=11, color=COL['ink'], loc='left')
    fig.tight_layout(); fig.savefig(OUT/f'그래프4_성격x시간대_{typ}.png', bbox_inches='tight'); plt.show()
print('그래프 저장 폴더:', OUT)